# PathLens-GNN — one method per run

Set `METHOD` to a directory name under `methods/`. The committed default is `ranking_diagnostics` with `STAGE=eval` (GPU rescore of heuristics + frozen PathLens: tie-break, zero-mass, degree-tertile MRR, visible vs all_positive filter). `STAGE=final` is refused. Do not open the sealed test.

The branch must already be on GitHub (`GIT_REF`). Enable Internet. Accelerator: GPU T4. Download one file: `/kaggle/working/pathlens-stage-output.zip` (`metrics.json` plus `figures/`).

In [ ]:
METHOD = "ranking_diagnostics"  # folder name under methods/
STAGE = "eval"  # smoke | train | eval | final — eval writes the tie/filter audit
GIT_REF = "research/ranking-loss"
RESUME_ARCHIVE = None
FINAL_TEST_TOKEN = ""
DEVICE = "cuda:0"


In [ ]:
import os
import pathlib
import subprocess
import sys

REPO = pathlib.Path("/kaggle/working/PathLens-GNN")
if not REPO.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--filter=blob:none",
            "https://github.com/aryonmt/PathLens-GNN.git",
            str(REPO),
        ],
        check=True,
    )
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)
repository_source = str(REPO / "src")
legacy_source = str(REPO / "legacy2" / "src")
if repository_source not in sys.path:
    sys.path.insert(0, repository_source)
if legacy_source not in sys.path:
    sys.path.insert(0, legacy_source)
print(f"METHOD={METHOD} STAGE={STAGE} DEVICE={DEVICE}")


In [ ]:
from pathlens.runtime.runner import run_stage

print(f"DEVICE={DEVICE}")
try:
    import torch

    print(f"torch={torch.__version__} cuda={torch.cuda.is_available()} name={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
except ImportError:
    print("torch not imported")

result = run_stage(
    METHOD,
    STAGE,
    device=DEVICE,
    final_test_token=FINAL_TEST_TOKEN,
)
print(result["output_dir"])
print(result.get("archive"))
print(f"device={result['device']} cuda={result.get('environment', {}).get('cuda_name')}")
if result.get("diagnostic"):
    for method_id, filters in result["methods"].items():
        ranking = filters["all_positive"]["ranking"]
        ties = filters["all_positive"]["ties"]
        visible = filters["visible"]["ranking"]["strict_gt"]["mrr"]
        print(
            f"{method_id}: strict={ranking['strict_gt']['mrr']:.4f} "
            f"average={ranking['average']['mrr']:.4f} "
            f"random={ranking['random']['mrr']:.4f} "
            f"visible_strict={visible:.4f} "
            f"zero={ties['fraction_zero_target']:.3f} "
            f"mean_tied={ties['mean_tied_others']:.1f}"
        )
    for skipped_id, reason in result.get("skipped", {}).items():
        print(f"skipped {skipped_id}: {reason}")
else:
    ranking = result["filtered_ranking"]
    hard = result["classification"]["hard"]
    print(
        f"mrr={ranking['mrr']:.4f} hits@10={ranking['hits_at_10']:.4f} "
        f"ndcg@10={ranking['ndcg_at_10']:.4f} hard_auprc={hard['auprc']:.4f}"
    )


In [ ]:
from pathlib import Path

from pathlens.evaluation.figures import copy_figure_files, load_validation_report, write_validation_figures
from pathlens.runtime.runner import archive_run_dir

run_dir = Path(result["output_dir"])
report = load_validation_report(Path("runs/biosnap-dti-v2"))
written = write_validation_figures(report, Path("runs/biosnap-dti-v2/figures"))
copy_figure_files(written, run_dir / "figures")
archive = result.get("archive")
if archive:
    archive_run_dir(run_dir, archive)
print("models:", ", ".join(sorted(report["models"])))
print("archive:", archive)
for path in written:
    print(path)